# APF3 · Predicción de demanda diaria por producto · Paukeñito Pollos & Parrillas
**Curso:** Innovación y Transformación Digital (UTP, sección 35024) · **Grupo 9**

Problema ML (regresión): *Para la propietaria y el personal de cocina de Paukeñito, el modelo predecirá las unidades que se venderán de cada producto principal al día siguiente, con el propósito de planificar compras e insumos, medido mediante MAE (unidades por producto-día) frente a la media móvil de 7 días que hoy sirve de línea base.*

Estructura: 1) carga y trazabilidad · 2) coherencia de datos · 3) EDA · 4) limpieza y variables · 5) partición · 6) modelos · 7) evaluación (test una sola vez) · 8) interpretación · 9) exportación e integración.

In [1]:
# Dependencias (ver requirements.txt). En Colab: !pip install -q pandas numpy scikit-learn matplotlib joblib
import json, warnings, joblib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

SEED = 42                      # random_state fijo en todo el notebook
np.random.seed(SEED)
import os
os.makedirs("fig", exist_ok=True); os.makedirs("model", exist_ok=True)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.3


## 1. Carga y trazabilidad con el Contrato de Datos (APF2)

In [2]:
RUTA = "data/dataset_demanda_diaria.csv"   # salida de 01_construir_dataset.py (vista vw_dataset_demanda_diaria)
raw = pd.read_csv(RUTA, parse_dates=["fecha"])
print(raw.shape); print(raw.head())
print("\nOrigen de los registros:\n", raw["origen"].value_counts())
print("Periodo:", raw["fecha"].min().date(), "->", raw["fecha"].max().date(),
      "| dias con ventas:", raw["fecha"].nunique(), "| productos:", raw["producto_id"].nunique())

(6207, 9)
       fecha  dia_semana  producto_id                producto  \
0 2026-04-12           1            3        Oferta Paukeñito   
1 2026-04-12           1            3        Oferta Paukeñito   
2 2026-04-12           1            4      Oferta con Chorizo   
3 2026-04-12           1            4      Oferta con Chorizo   
4 2026-04-12           1            6  Oferta Leña a lo Pobre   

                    categoria tipo_venta  unidades_vendidas  monto_vendido  \
0  Pollos a la Leña - Ofertas     llevar                1.0           18.0   
1  Pollos a la Leña - Ofertas      local                2.0           36.0   
2  Pollos a la Leña - Ofertas     llevar                1.0           20.0   
3  Pollos a la Leña - Ofertas      local                2.0           40.0   
4  Pollos a la Leña - Ofertas     llevar                1.0           19.0   

     origen  
0  simulado  
1  simulado  
2  simulado  
3  simulado  
4  simulado  

Origen de los registros:
 origen
simulado    

**Tabla de trazabilidad con el Contrato de Datos de APF2** (se completa con las cifras obtenidas al ejecutar):

In [3]:
trazab = pd.DataFrame([
 ["Variable objetivo", "unidades_vendidas por producto y dia", "unidades_vendidas por producto y dia (top 12 productos, tipo_venta sumado)", "Se suma tipo_venta (y se completan con 0 los dias sin ventas) para evitar filas casi vacias; el grano del contrato se conserva en el CSV"],
 ["Campos", "fecha, dia_semana, producto_id, producto, tipo_venta, unidades_vendidas, monto_vendido", ", ".join(raw.columns), "Se agregan origen (real|simulado), exigida por la consigna, y categoria (catalogo del MVP), para el analisis por categoria que el docente pidio para el dashboard"],
 ["Fuente", "ventas y detalle_ventas del MVP (vista vw_dataset_demanda_diaria)", "Respaldo polleria_db.sql del repositorio del MVP (misma estructura)", "Equivalente a la vista; permite reproducir el dataset desde el repositorio"],
 ["Volumen", "Minimo de 100 filas y 4 semanas", f"{len(raw):,} filas y 26 semanas (3,472 ventas completadas)", "Se cumple el minimo; sin cambio"],
 ["Calidad", "Solo completadas; sin nombre, DNI, telefono ni direccion", "Solo completadas; sin datos personales", "Sin cambio"],
 ["Particion", "Temporal: train 12/04-30/08, val 31/08-20/09, test 21/09-11/10", "Misma particion temporal", "Sin cambio"],
], columns=["Elemento", "Contrato APF2", "Dataset APF3", "Cambio / justificacion"])
trazab.to_csv("data/tabla_trazabilidad.csv", index=False); print(trazab.to_string(index=False))

         Elemento                                                                          Contrato APF2                                                                                              Dataset APF3                                                                                                                                            Cambio / justificacion
Variable objetivo                                                   unidades_vendidas por producto y dia                                unidades_vendidas por producto y dia (top 12 productos, tipo_venta sumado)                          Se suma tipo_venta (y se completan con 0 los dias sin ventas) para evitar filas casi vacias; el grano del contrato se conserva en el CSV
           Campos fecha, dia_semana, producto_id, producto, tipo_venta, unidades_vendidas, monto_vendido fecha, dia_semana, producto_id, producto, categoria, tipo_venta, unidades_vendidas, monto_vendido, origen Se agregan origen (real|simulado), exigida 

### Diccionario de datos
| Variable | Tipo | Unidad | Descripcion | Rango valido |
|---|---|---|---|---|
| fecha | date | dia | Dia de la venta (hora de Lima) | 2026-04-12 a 2026-10-11 |
| dia_semana | int | - | 1 = domingo ... 7 = sabado (contrato APF2) | 1-7 |
| producto_id / producto | int / texto | - | Identificador y nombre del producto | catalogo del MVP |
| tipo_venta | texto | - | local, llevar o delivery | 3 valores |
| unidades_vendidas | decimal | unidades | Unidades vendidas del producto ese dia | > 0 |
| monto_vendido | decimal | soles (S/) | Monto vendido del producto ese dia | > 0 |
| origen | texto | - | real o simulado | simulado |

Unidad de observacion del modelo: **un producto en un dia** (fila = producto-dia).

### Procedimiento de simulacion y origen de los datos
Los registros **no provienen de ventas reales de la propietaria**: son datos de prueba autorizados para el curso. Lo que se puede verificar:
- Fuente unica y versionada: el archivo `02_seed_paukenito_smp.sql` (version v5, repositorio `sistema-polleria`, commit 56134f0). Es un archivo fijo, sin aleatoriedad al cargarse; por eso el dataset es identico en cada reconstruccion.
- Descripcion de la propia semilla (cabecera del archivo): periodo 11/04-11/10/2026, 3,493 ventas completadas, S/ 157,305.50 en total (aprox. S/ 5,984 por semana), costo de delivery por zona de S/ 3 a S/ 8 y pedidos de personas no registradas. Estas cifras se contrastaron con la base cargada y coinciden.
- Catalogo, categorias y precios: los del MVP (92 productos con venta en el periodo).
- Del periodo del contrato (12/04-11/10) quedan 3,472 ventas completadas; el CSV de modelado tiene una fila por fecha, producto y tipo de venta.

**Limitaciones declaradas:** el repositorio no incluye el script que genero la semilla, por lo que la calibracion del volumen semanal es un supuesto de simulacion que debe contrastarse con los registros de la propietaria. Los patrones que se observan en el EDA (picos de fin de semana y fechas especiales) son propiedades de estos datos de prueba y no prueban que el negocio real se comporte igual. Por eso los resultados se presentan como validacion tecnica del enfoque, no como medicion del negocio real.

In [4]:
# Punto de conexion con ventas reales: si la propietaria registra ventas en el MVP, se exporta la vista con la misma estructura
# a data/ventas_reales.csv (origen = 'real') y esta celda compara las distribuciones antes de reentrenar.
import os
if os.path.exists("data/ventas_reales.csv"):
    real = pd.read_csv("data/ventas_reales.csv", parse_dates=["fecha"]); real["origen"] = "real"
    cmp_o = pd.concat([raw.assign(origen="simulado"), real]).groupby("origen").agg(
        filas=("fecha", "size"), dias=("fecha", "nunique"), u_dia=("unidades_vendidas", lambda s: s.sum()), media_fila=("unidades_vendidas", "mean"))
    cmp_o["u_dia"] = cmp_o.u_dia / pd.concat([raw.assign(origen="simulado"), real]).groupby("origen").fecha.nunique()
    print(cmp_o.round(2).to_string())
else:
    print("data/ventas_reales.csv no existe: en esta entrega todos los registros son origen = simulado (no hay mezcla con datos reales).")

data/ventas_reales.csv no existe: en esta entrega todos los registros son origen = simulado (no hay mezcla con datos reales).


## 2. Coherencia de los datos (ficha de 7 criterios)

In [5]:
resultados = []
# 1. Escala del negocio: monto diario y mensual acorde con el tamano declarado en APF1 (tienda unica, 8 colaboradores)
monto_dia = raw.groupby("fecha")["monto_vendido"].sum()
resultados.append(("Escala del negocio", f"Monto diario medio S/ {monto_dia.mean():.0f} (min {monto_dia.min():.0f}, max {monto_dia.max():.0f}), unos S/ {round(monto_dia.mean() * 30, -3):,.0f} al mes, para una tienda unica de 8 colaboradores. APF1 no declara cifras de ventas; escala plausible para ese tamano, a contrastar con la propietaria con ventas reales"))
# 2. Rangos y unidades
viol = int(((raw["unidades_vendidas"] <= 0) | (raw["monto_vendido"] <= 0) | (~raw["dia_semana"].between(1, 7))).sum())
fut = int((raw["fecha"] > "2026-10-11").sum())
resultados.append(("Rangos y unidades", f"Violaciones de rango: {viol}; fechas futuras: {fut}; nulos: {int(raw.isna().sum().sum())}"))
# 3. Consistencia logica y temporal
dup = int(raw.duplicated(["fecha", "producto_id", "tipo_venta"]).sum())
chk_dia = int((((raw["fecha"].dt.dayofweek + 1) % 7 + 1) != raw["dia_semana"]).sum())
resultados.append(("Consistencia logica/temporal", f"Duplicados por (fecha, producto, tipo): {dup}; dia_semana incoherente con fecha: {chk_dia}; tipos permitidos: {sorted(raw['tipo_venta'].unique())}"))
# 4. Relaciones plausibles (se verifican en el EDA)
por_dia = raw.groupby(["fecha"])["unidades_vendidas"].sum().to_frame("u").assign(d=lambda t: t.index.dayofweek)
dom = por_dia[por_dia.d == 6].u.mean(); lab = por_dia[por_dia.d.isin([0, 1, 2, 3])].u.mean()
esp = ["2026-05-10", "2026-06-21", "2026-07-19", "2026-07-28", "2026-07-29"]
u_esp = por_dia.loc[por_dia.index.isin(pd.to_datetime(esp)), "u"].mean(); u_nor = por_dia.loc[~por_dia.index.isin(pd.to_datetime(esp)), "u"].mean()
resultados.append(("Relaciones plausibles", f"Domingo {dom:.0f} u./dia vs lun-jue {lab:.0f} u./dia; fechas especiales {u_esp:.0f} vs normales {u_nor:.0f} u./dia"))
# 6. Coherencia con APF2
cols_apf2 = {"fecha", "dia_semana", "producto_id", "producto", "tipo_venta", "unidades_vendidas", "monto_vendido"}
resultados.append(("Coherencia con APF2", f"Campos del contrato presentes: {cols_apf2.issubset(raw.columns)} (mas columnas origen y categoria)"))
# 7. Mezcla real + simulado
resultados.append(("Mezcla real + simulado", f"Origen: {raw['origen'].value_counts().to_dict()} -> no hay mezcla; todo simulado, no se presenta como real"))
ficha = pd.DataFrame(resultados, columns=["Criterio", "Verificacion ejecutada"])
ficha.to_csv("data/ficha_coherencia.csv", index=False); print(ficha.to_string(index=False))
assert viol == 0 and fut == 0 and dup == 0 and chk_dia == 0, "Datos incoherentes: revisar"
# El criterio 5 (senal ni nula ni perfecta) se audita despues del modelado, en la seccion 7.

                    Criterio                                                                                                                                                                                                                Verificacion ejecutada
          Escala del negocio Monto diario medio S/ 864 (min 426, max 2412), unos S/ 26,000 al mes, para una tienda unica de 8 colaboradores. APF1 no declara cifras de ventas; escala plausible para ese tamano, a contrastar con la propietaria con ventas reales
           Rangos y unidades                                                                                                                                                                                  Violaciones de rango: 0; fechas futuras: 0; nulos: 0
Consistencia logica/temporal                                                                                                     Duplicados por (fecha, producto, tipo): 0; dia_semana incoherente con fecha: 0; tipos permitid

## 3. Construccion de la tabla de modelado y EDA

In [6]:
# Se modelan los 12 productos con mas unidades (concentran la demanda y evitan filas casi vacias).
top = raw.groupby("producto_id")["unidades_vendidas"].sum().nlargest(12).index
nombres = raw.drop_duplicates("producto_id").set_index("producto_id")["producto"]
share = raw[raw.producto_id.isin(top)].unidades_vendidas.sum() / raw.unidades_vendidas.sum()
print(f"Top 12 productos = {share:.1%} de las unidades vendidas")

dias = pd.date_range("2026-04-12", "2026-10-11")
panel = (raw[raw.producto_id.isin(top)].groupby(["fecha", "producto_id"])["unidades_vendidas"].sum()
         .unstack(fill_value=0).reindex(dias, fill_value=0))      # dia sin ventas del producto = 0 unidades
panel.index.name = "fecha"
y_long = panel.stack().rename("unidades").reset_index()
print("Filas de modelado:", len(y_long), "| ceros:", f"{(y_long.unidades == 0).mean():.1%}")
print(y_long.unidades.describe().round(2).to_string())

Top 12 productos = 39.5% de las unidades vendidas
Filas de modelado: 2196 | ceros: 32.2%
count    2196.0
mean        1.6
std         1.7
min         0.0
25%         0.0
50%         1.0
75%         2.0
max        14.0


In [7]:
# EDA 1: estacionalidad semanal y fechas especiales
tot = panel.sum(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
tot.plot(ax=ax[0], lw=1); ax[0].set_title("Unidades diarias (12 productos principales)"); ax[0].set_xlabel("")
for f in esp: ax[0].axvline(pd.Timestamp(f), color="tab:red", alpha=.35, lw=1)
nom = ["Lun", "Mar", "Mie", "Jue", "Vie", "Sab", "Dom"]
tot.groupby(tot.index.dayofweek).mean().rename(dict(enumerate(nom))).plot.bar(ax=ax[1], color="tab:blue")
ax[1].set_title("Promedio de unidades por dia de la semana"); ax[1].set_xlabel("")
plt.tight_layout(); plt.savefig("fig/eda1_estacionalidad.png", dpi=150); plt.close()
print(tot.groupby(tot.index.dayofweek).mean().round(1).rename(dict(enumerate(nom))).to_dict())
print("Maximo diario:", tot.idxmax().date(), int(tot.max()), "u.")

{'Lun': 16.4, 'Mar': 15.7, 'Mie': 18.6, 'Jue': 18.4, 'Vie': 19.4, 'Sab': 21.6, 'Dom': 24.1}
Maximo diario: 2026-07-19 52 u.


In [8]:
# EDA 2: distribucion del target (sesgo, ceros) y demanda por producto
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
y_long.unidades.plot.hist(bins=range(0, int(y_long.unidades.max()) + 2), ax=ax[0], color="tab:green", edgecolor="white")
ax[0].set_title("Distribucion de unidades por producto-dia"); ax[0].set_xlabel("unidades")
panel.rename(columns=nombres).sum().sort_values().plot.barh(ax=ax[1], color="tab:orange")
ax[1].set_title("Unidades totales por producto (12 principales)")
plt.tight_layout(); plt.savefig("fig/eda2_distribucion.png", dpi=150); plt.close()
print("Sesgo:", round(float(y_long.unidades.skew()), 2), "| mediana:", float(y_long.unidades.median()), "| p95:", float(y_long.unidades.quantile(.95)))
print("Correlacion unidades de hoy con las de hace 7 dias (por producto, media):",
      round(float(np.nanmean([panel[c].autocorr(7) for c in panel.columns])), 2))

Sesgo: 1.52 | mediana: 1.0 | p95: 5.0
Correlacion unidades de hoy con las de hace 7 dias (por producto, media): 0.02


**Conclusiones del EDA (se redactan en el informe con las cifras impresas arriba):** la demanda tiene patron semanal y picos en fechas especiales, y el target es discreto, sesgado y con muchos ceros. Por eso se usan rezagos de 1 y 7 dias, el dia de la semana y una marca de fecha especial, y se evalua con MAE (robusto a los picos) ademas de RMSE y R².

## 4. Limpieza y variables
Sin nulos ni duplicados (ver ficha). Los outliers (picos de fechas especiales) **no se eliminan**: son demanda real del negocio y se explican con la variable `fecha_especial`. Variable derivada: promedios moviles y marca de fecha especial; todas usan solo informacion hasta el dia anterior (sin fuga).

In [9]:
ESPECIALES = pd.to_datetime(["2026-05-10",   # Dia de la Madre
                             "2026-06-21",   # Dia del Padre
                             "2026-07-19",   # Dia del Pollo a la Brasa (3er domingo de julio)
                             "2026-07-28", "2026-07-29"])  # Fiestas Patrias
frames = []
for pid in panel.columns:
    s = panel[pid]
    f = pd.DataFrame({"fecha": panel.index, "producto_id": pid, "y": s.values})
    f["lag_1"] = s.shift(1).values                       # ayer
    f["lag_7"] = s.shift(7).values                       # mismo dia de la semana pasada
    f["media_7"] = s.shift(1).rolling(7).mean().values    # linea base y variable derivada (solo pasado)
    f["media_14"] = s.shift(1).rolling(14).mean().values
    frames.append(f)
d = pd.concat(frames, ignore_index=True)
d["dia_semana"] = d.fecha.dt.dayofweek.astype(str)
d["fin_de_semana"] = d.fecha.dt.dayofweek.isin([5, 6]).astype(int)
d["fecha_especial"] = d.fecha.isin(ESPECIALES).astype(int)
d["origen"] = "simulado"
d = d.dropna().reset_index(drop=True)       # primeras 14 fechas sin historial suficiente
print(d.shape); print(d.head(3).to_string())

(2028, 11)
       fecha  producto_id    y  lag_1  lag_7   media_7  media_14 dia_semana  fin_de_semana  fecha_especial    origen
0 2026-04-26            3  0.0    3.0    5.0  2.714286  2.785714          6              1               0  simulado
1 2026-04-27            3  1.0    0.0    2.0  2.000000  2.571429          0              0               0  simulado
2 2026-04-28            3  0.0    1.0    3.0  1.857143  2.428571          1              0               0  simulado


## 5. Particion temporal y prevencion de fuga
Particion cronologica fijada en el Contrato de Datos (sin mezcla aleatoria): **train** hasta el 30/08, **validacion** 31/08-20/09, **test** 21/09-11/10. El escalado y la codificacion se ajustan solo con train dentro de un `Pipeline`.

In [10]:
NUM = ["lag_1", "lag_7", "media_7", "media_14"]
BIN = ["fin_de_semana", "fecha_especial"]
CAT = ["dia_semana", "producto_id"]
d["producto_id"] = d["producto_id"].astype(str)
FEATS = NUM + BIN + CAT
tr = d[d.fecha <= "2026-08-30"]; va = d[(d.fecha > "2026-08-30") & (d.fecha <= "2026-09-20")]; te = d[d.fecha > "2026-09-20"]
Xtr, ytr, Xva, yva, Xte, yte = tr[FEATS], tr.y, va[FEATS], va.y, te[FEATS], te.y
print(f"train {len(tr)} filas ({tr.fecha.min().date()}-{tr.fecha.max().date()}) | val {len(va)} ({va.fecha.min().date()}-{va.fecha.max().date()}) | test {len(te)} ({te.fecha.min().date()}-{te.fecha.max().date()})")

def prep(escalar=True):
    num = StandardScaler() if escalar else "passthrough"
    return ColumnTransformer([("num", num, NUM), ("bin", "passthrough", BIN),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
def mets(y, p): return dict(MAE=mean_absolute_error(y, p), RMSE=mean_squared_error(y, p) ** .5, R2=r2_score(y, p))

# Control de fuga: lag_1 debe ser exactamente la demanda real del dia anterior (nunca la del dia a predecir)
ayer = y_long.assign(fecha=y_long.fecha + pd.Timedelta(days=1), producto_id=y_long.producto_id.astype(str)).rename(columns={"unidades": "ayer_real"})
chk = d.merge(ayer, on=["fecha", "producto_id"])
assert (chk.lag_1 == chk.ayer_real).all() and len(chk) == len(d)
print("Control de fuga OK: las variables de rezago usan solo dias anteriores al que se predice.")

train 1524 filas (2026-04-26-2026-08-30) | val 252 (2026-08-31-2026-09-20) | test 252 (2026-09-21-2026-10-11)
Control de fuga OK: las variables de rezago usan solo dias anteriores al que se predice.


## 6. Modelado: linea base y modelos de la Unidad 3

In [11]:
# Linea base: media movil de 7 dias (regla que ya usa el MVP) y modelo trivial (media de entrenamiento)
base_ma7 = lambda X: X["media_7"].values
dummy = Pipeline([("p", prep(False)), ("m", DummyRegressor(strategy="mean"))]).fit(Xtr, ytr)
# Referencia mas exigente: promedio historico de cada producto (calculado solo con train)
media_prod = ytr.groupby(Xtr["producto_id"]).mean()
base_prod = lambda X: X["producto_id"].map(media_prod).fillna(ytr.mean()).values
print("Validacion | Media historica por producto:", {k: round(v, 3) for k, v in mets(yva, base_prod(Xva)).items()})
print("Validacion | MA7:", {k: round(v, 3) for k, v in mets(yva, base_ma7(Xva)).items()},
      "| Dummy:", {k: round(v, 3) for k, v in mets(yva, dummy.predict(Xva)).items()})

Validacion | Media historica por producto: {'MAE': 1.182, 'RMSE': 1.497, 'R2': 0.135}
Validacion | MA7: {'MAE': 1.274, 'RMSE': 1.635, 'R2': -0.032} | Dummy: {'MAE': 1.356, 'RMSE': 1.617, 'R2': -0.01}


In [12]:
# Busqueda en grilla con validacion cruzada cronologica SOLO sobre train (TimeSeriesSplit, 4 particiones)
cv = TimeSeriesSplit(n_splits=4)
espacios = {
  "Ridge (regresion regularizada)": (Pipeline([("p", prep(True)), ("m", Ridge(random_state=SEED))]),
        {"m__alpha": [1, 10, 50, 100, 300, 1000]}),
  "Random Forest": (Pipeline([("p", prep(False)), ("m", RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))]),
        {"m__max_depth": [2, 3, 5, 8], "m__min_samples_leaf": [5, 10, 20, 40]}),
  "Gradient Boosting": (Pipeline([("p", prep(False)), ("m", GradientBoostingRegressor(random_state=SEED, subsample=0.8))]),
        {"m__n_estimators": [50, 100, 200], "m__learning_rate": [0.01, 0.03, 0.1], "m__max_depth": [1, 2, 3]}),
}
mejores, tabla = {}, []
for nombre, (pipe, grid) in espacios.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1).fit(Xtr, ytr)
    m = gs.best_estimator_; mejores[nombre] = m
    mtr, mva = mets(ytr, m.predict(Xtr)), mets(yva, m.predict(Xva))
    tabla.append([nombre, gs.best_params_, -gs.best_score_, mtr["MAE"], mva["MAE"], mva["RMSE"], mva["R2"]])
    print(f"{nombre}: mejores parametros {gs.best_params_} | MAE CV {-gs.best_score_:.3f}")
sel = pd.DataFrame(tabla, columns=["Modelo", "Hiperparametros", "MAE_CV_train", "MAE_train", "MAE_val", "RMSE_val", "R2_val"])
sel["Brecha_train_val"] = sel.MAE_val - sel.MAE_train
sel.to_csv("data/seleccion_validacion.csv", index=False)
print(sel.drop(columns="Hiperparametros").round(3).to_string(index=False))
ma7_val = mets(yva, base_ma7(Xva)); print("Linea base MA7 en validacion:", {k: round(v, 3) for k, v in ma7_val.items()})

Ridge (regresion regularizada): mejores parametros {'m__alpha': 100} | MAE CV 1.218


Random Forest: mejores parametros {'m__max_depth': 3, 'm__min_samples_leaf': 5} | MAE CV 1.198


Gradient Boosting: mejores parametros {'m__learning_rate': 0.1, 'm__max_depth': 1, 'm__n_estimators': 50} | MAE CV 1.202
                        Modelo  MAE_CV_train  MAE_train  MAE_val  RMSE_val  R2_val  Brecha_train_val
Ridge (regresion regularizada)         1.218      1.170    1.198     1.513   0.116             0.028
                 Random Forest         1.198      1.152    1.217     1.536   0.089             0.065
             Gradient Boosting         1.202      1.167    1.219     1.536   0.089             0.051
Linea base MA7 en validacion: {'MAE': 1.274, 'RMSE': 1.635, 'R2': -0.032}


**Seleccion y control de sobreajuste.** El modelo se elige por MAE en validacion (no en test). El sobreajuste se controla con regularizacion (`alpha` en Ridge), profundidad maxima y hoja minima (Random Forest) y `subsample` con profundidad 2-3 (Gradient Boosting); la brecha train vs validacion se reporta en la tabla.

In [13]:
ELEGIDO = sel.loc[sel.MAE_val.idxmin(), "Modelo"]
print("Modelo seleccionado por validacion:", ELEGIDO)
# Reentrenamiento del modelo elegido con train + validacion (mismos hiperparametros) antes de tocar el test
from sklearn.base import clone
final = clone(mejores[ELEGIDO]).fit(pd.concat([Xtr, Xva]), pd.concat([ytr, yva]))
print("Modelo final entrenado con", len(Xtr) + len(Xva), "filas (hasta el 20/09)")

Modelo seleccionado por validacion: Ridge (regresion regularizada)
Modelo final entrenado con 1776 filas (hasta el 20/09)


### 6.1 Experimentos adicionales de variables (solo validacion)
Para revisar si la senal debil se debia a pocas variables, se probaron conjuntos ampliados con regresion regularizada y regresion de Poisson (adecuada para conteos). Se eligen **solo con validacion**; el test no se toca.

In [14]:
dx = d.copy()
tot_all = panel.sum(axis=1)
dx["tot_lag_7"] = dx.fecha.map(tot_all.shift(7)); dx["tot_media_7"] = dx.fecha.map(tot_all.shift(1).rolling(7).mean())
dx["tot_media_28"] = dx.fecha.map(tot_all.shift(1).rolling(28).mean())
dx["media_28"] = np.nan
for pid in panel.columns:
    m = dx.producto_id == str(pid); dx.loc[m, "media_28"] = dx.loc[m, "fecha"].map(panel[pid].shift(1).rolling(28).mean())
dx["dia_mes"] = dx.fecha.dt.day; dx["quincena"] = dx.fecha.dt.day.isin([14, 15, 16, 29, 30, 31, 1]).astype(int)
dx = dx.dropna().reset_index(drop=True)
trx = dx[dx.fecha <= "2026-08-30"]; vax = dx[(dx.fecha > "2026-08-30") & (dx.fecha <= "2026-09-20")]
def prep_x(n, b, c): return ColumnTransformer([("n", StandardScaler(), n), ("b", "passthrough", b), ("c", OneHotEncoder(handle_unknown="ignore"), c)])
from sklearn.linear_model import PoissonRegressor
conj = {"A. Variables base (las usadas)": (NUM, BIN, CAT),
        "B. + totales de la tienda (lag 7, media 7)": (NUM + ["tot_lag_7", "tot_media_7"], BIN, CAT),
        "C. + medias de 28 dias": (NUM + ["media_28", "tot_media_28"], BIN, CAT),
        "D. + dia del mes y quincena": (NUM + ["media_28", "tot_lag_7", "tot_media_7", "dia_mes"], BIN + ["quincena"], CAT)}
filas = []
for nom, (n, b, c) in conj.items():
    for mod, pipe, g in [("Ridge", Pipeline([("p", prep_x(n, b, c)), ("m", Ridge())]), {"m__alpha": [10, 100, 300, 1000]}),
                         ("Poisson", Pipeline([("p", prep_x(n, b, c)), ("m", PoissonRegressor(max_iter=500))]), {"m__alpha": [0.1, 1, 10, 50]})]:
        gs = GridSearchCV(pipe, g, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", n_jobs=-1).fit(trx[n + b + c], trx.y)
        filas.append([nom, mod, -gs.best_score_, mean_absolute_error(vax.y, gs.predict(vax[n + b + c]))])
exp = pd.DataFrame(filas, columns=["Conjunto de variables", "Modelo", "MAE_CV_train", "MAE_val"])
ref_prod = mean_absolute_error(vax.y, vax.producto_id.map(trx.groupby("producto_id").y.mean()))
exp.to_csv("data/experimentos_variables_val.csv", index=False)
print(exp.round(3).to_string(index=False)); print("Referencia en validacion: media historica por producto =", round(ref_prod, 3), "| MA7 =", round(mean_absolute_error(vax.y, vax.media_7), 3))
print("Mejor MAE_val de los experimentos:", round(exp.MAE_val.min(), 3), "| mejor modelo base (seccion 6):", round(sel.MAE_val.min(), 3))

                     Conjunto de variables  Modelo  MAE_CV_train  MAE_val
            A. Variables base (las usadas)   Ridge         1.241    1.201
            A. Variables base (las usadas) Poisson         1.248    1.245
B. + totales de la tienda (lag 7, media 7)   Ridge         1.241    1.211
B. + totales de la tienda (lag 7, media 7) Poisson         1.247    1.252
                    C. + medias de 28 dias   Ridge         1.223    1.203
                    C. + medias de 28 dias Poisson         1.226    1.204
               D. + dia del mes y quincena   Ridge         1.232    1.201
               D. + dia del mes y quincena Poisson         1.236    1.202
Referencia en validacion: media historica por producto = 1.19 | MA7 = 1.274
Mejor MAE_val de los experimentos: 1.201 | mejor modelo base (seccion 6): 1.198


**Lectura:** ningun conjunto ampliado mejora de forma material la validacion (las diferencias son de centesimas de unidad, dentro del ruido). Por eso se **conservan las variables base**: agregar variables sin ganancia solo sumaria complejidad. A nivel producto-dia la demanda es un conteo pequeno (media ~1.6 u.) con mucho ruido; en 7.1 se revisa si agregar al total diario cambia la conclusion.

## 7. Evaluacion en el conjunto de prueba (se usa una sola vez)

In [15]:
TEST_USADO = False
assert not TEST_USADO, "El test ya se uso: no repetir para elegir modelos"
res = {"Linea base: media movil 7 dias": mets(yte, base_ma7(Xte)),
       "Modelo trivial: media de entrenamiento": mets(yte, dummy.predict(Xte)),
       "Referencia exigente: media historica por producto": mets(yte, base_prod(Xte))}
for nombre, m in mejores.items():
    res[nombre + (" (SELECCIONADO)" if nombre == ELEGIDO else "")] = mets(yte, m.predict(Xte)) if nombre != ELEGIDO else mets(yte, final.predict(Xte))
TEST_USADO = True
tabla_test = pd.DataFrame(res).T.round(3); tabla_test.to_csv("data/metricas_test.csv")
print(tabla_test.to_string())
mae_base = res["Linea base: media movil 7 dias"]["MAE"]; mae_mod = mets(yte, final.predict(Xte))["MAE"]
print(f"\nMejora del MAE frente a la linea base: {(mae_base - mae_mod) / mae_base:.1%}")
r2 = mets(yte, final.predict(Xte))["R2"]
assert r2 < 0.98, "R2 >= 0.98: auditar fuga o simulacion trivial"
print(f"R2 del modelo = {r2:.3f} (< 0.98: la senal es explicable y no perfecta)")

                                                     MAE   RMSE     R2
Linea base: media movil 7 dias                     1.206  1.638  0.045
Modelo trivial: media de entrenamiento             1.286  1.678 -0.002
Referencia exigente: media historica por producto  1.117  1.512  0.187
Ridge (regresion regularizada) (SELECCIONADO)      1.142  1.540  0.156
Random Forest                                      1.136  1.547  0.149
Gradient Boosting                                  1.146  1.545  0.151

Mejora del MAE frente a la linea base: 5.3%
R2 del modelo = 0.156 (< 0.98: la senal es explicable y no perfecta)


In [16]:
# Analisis de errores: predicho vs real, residuos y 3 casos donde el modelo falla
pred = final.predict(Xte); te2 = te.assign(pred=pred, error=lambda t: t.y - t.pred, abs_err=lambda t: (t.y - t.pred).abs())
te2["producto"] = te2.producto_id.astype(int).map(nombres)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(te2.y, te2.pred, alpha=.5); lim = max(te2.y.max(), te2.pred.max()) + 1
ax[0].plot([0, lim], [0, lim], "r--"); ax[0].set_xlabel("real"); ax[0].set_ylabel("predicho"); ax[0].set_title("Predicho vs real (test)")
ax[1].hist(te2.error, bins=20, color="tab:purple", edgecolor="white"); ax[1].set_title("Residuos (real - predicho)")
plt.tight_layout(); plt.savefig("fig/errores.png", dpi=150); plt.close()
peores = te2.sort_values("abs_err", ascending=False).head(3)[["fecha", "producto", "dia_semana", "y", "pred", "lag_1", "lag_7", "media_7"]]
peores["pred"] = peores.pred.round(2); print(peores.to_string(index=False)); peores.to_csv("data/peores_casos.csv", index=False)
print("Sesgo medio (real - predicho):", round(float(te2.error.mean()), 3), "| subestima en", f"{(te2.error > 0).mean():.0%} de los casos")
fig, ax = plt.subplots(figsize=(7, 3.5))
cmp = te2.groupby("fecha")[["y", "pred"]].sum(); cmp["ma7"] = te.groupby("fecha").media_7.sum()
cmp.plot(ax=ax); ax.set_title("Total diario, 12 productos: real, modelo y media movil 7 d. (test)"); plt.tight_layout(); plt.savefig("fig/serie_test.png", dpi=150); plt.close()

     fecha           producto dia_semana    y  pred  lag_1  lag_7  media_7
2026-10-07   Inca Kola 500 ml          2 10.0  2.91    0.0    4.0 4.142857
2026-09-26   Inca Kola 500 ml          5  9.0  2.84    4.0    2.0 2.571429
2026-09-21 Oferta con Chorizo          0  7.0  1.14    1.0    0.0 1.142857
Sesgo medio (real - predicho): -0.0 | subestima en 40% de los casos


In [17]:
# Incertidumbre de la mejora: bootstrap por dias (los productos de un mismo dia no son independientes)
te3 = te.assign(e_mod=(te.y - final.predict(Xte)).abs(), e_ma7=(te.y - te.media_7).abs(), e_prod=(te.y - base_prod(Xte)).abs())
g = te3.groupby("fecha")[["e_mod", "e_ma7", "e_prod"]].mean(); rng = np.random.default_rng(SEED)
idx = [rng.choice(len(g), size=len(g), replace=True) for _ in range(5000)]
def ic(col):
    rel = np.array([1 - g.e_mod.values[i].mean() / g[col].values[i].mean() for i in idx])
    return np.percentile(rel, [2.5, 97.5]) * 100, (rel > 0).mean()
for col, nom in [("e_ma7", "media movil 7 dias"), ("e_prod", "media historica por producto")]:
    (lo, hi), pr = ic(col)
    print(f"Mejora relativa del MAE frente a {nom}: IC95% [{lo:.1f} %, {hi:.1f} %] | P(mejora > 0) = {pr:.2f}")
print("Dias de prueba en que el modelo supera a la media movil:", int((g.e_mod < g.e_ma7).sum()), "de", len(g))

Mejora relativa del MAE frente a media movil 7 dias: IC95% [0.8 %, 9.8 %] | P(mejora > 0) = 0.99
Mejora relativa del MAE frente a media historica por producto: IC95% [-6.2 %, 1.3 %] | P(mejora > 0) = 0.13
Dias de prueba en que el modelo supera a la media movil: 14 de 21


### 7.1 Prediccion del total diario (12 productos): revision a nivel agregado
Misma disciplina: los candidatos se comparan en **validacion**, se reentrena con train + validacion y el test (21 dias) se evalua una sola vez. Con solo 21 dias de prueba el resultado es indicativo, no concluyente.

**Lectura (se redacta con las cifras impresas abajo):** el patron semanal existe pero es suave frente al ruido diario, y no se repite igual en las 3 semanas de validacion; por eso la media de entrenamiento resulta competitiva y el intervalo de la mejora sobre la media movil incluye el cero. No se afirma una mejora significativa a nivel agregado.

In [18]:
t = tot_all.to_frame("y")
t["l7"] = tot_all.shift(7); t["m7"] = tot_all.shift(1).rolling(7).mean()
for i in range(7): t[f"d{i}"] = (t.index.dayofweek == i).astype(int)
t["esp"] = t.index.isin(ESPECIALES).astype(int); t = t.dropna()
FT = ["l7", "m7", "esp"] + [f"d{i}" for i in range(7)]
ttr, tva, tte = t[:"2026-08-30"], t["2026-08-31":"2026-09-20"], t["2026-09-21":]
def dow_mean(fit, ev): return ev.index.dayofweek.map(fit.groupby(fit.index.dayofweek).y.mean()).values
cand_val = {"Media movil 7 dias": tva.m7.values, "Media de entrenamiento": np.repeat(ttr.y.mean(), len(tva)),
            "Media por dia de la semana": dow_mean(ttr, tva)}
for a in [1, 10, 100]: cand_val[f"Ridge alpha={a}"] = Ridge(alpha=a).fit(ttr[FT], ttr.y).predict(tva[FT])
val_t = pd.Series({k: mean_absolute_error(tva.y, v) for k, v in cand_val.items()}); print("MAE en validacion (total diario, unidades):"); print(val_t.round(2).to_string())
mejor_t = val_t.idxmin(); print("Elegido por validacion:", mejor_t)
full = pd.concat([ttr, tva])
def pred_test(nombre):
    if nombre == "Media movil 7 dias": return tte.m7.values
    if nombre == "Media de entrenamiento": return np.repeat(full.y.mean(), len(tte))
    if nombre == "Media por dia de la semana": return dow_mean(full, tte)
    return Ridge(alpha=float(nombre.split("=")[1])).fit(full[FT], full.y).predict(tte[FT])
test_t = pd.Series({"Media movil 7 dias": mean_absolute_error(tte.y, pred_test("Media movil 7 dias")),
                    "Media de entrenamiento": mean_absolute_error(tte.y, pred_test("Media de entrenamiento")),
                    f"Elegido ({mejor_t})": mean_absolute_error(tte.y, pred_test(mejor_t))})
print("\nMAE en test (total diario, 21 dias; media real del periodo:", round(tte.y.mean(), 1), "u/dia):"); print(test_t.round(2).to_string())
e_m, e_b = np.abs(tte.y.values - pred_test(mejor_t)), np.abs(tte.y.values - tte.m7.values)
rng2 = np.random.default_rng(SEED); ii = [rng2.choice(len(e_m), len(e_m)) for _ in range(5000)]
rel = np.array([1 - e_m[i].mean() / e_b[i].mean() for i in ii])
print(f"Mejora del MAE frente a la media movil 7 dias: {1 - e_m.mean() / e_b.mean():.1%} | IC95% [{np.percentile(rel, 2.5) * 100:.1f} %, {np.percentile(rel, 97.5) * 100:.1f} %]")
pd.concat([val_t.rename("MAE_val"), test_t.rename("MAE_test")], axis=1).to_csv("data/total_diario.csv")

MAE en validacion (total diario, unidades):
Media movil 7 dias            6.35
Media de entrenamiento        5.59
Media por dia de la semana    6.05
Ridge alpha=1                 5.86
Ridge alpha=10                5.93
Ridge alpha=100               5.98
Elegido por validacion: Media de entrenamiento

MAE en test (total diario, 21 dias; media real del periodo: 18.7 u/dia):
Media movil 7 dias                  4.50
Media de entrenamiento              4.16
Elegido (Media de entrenamiento)    4.16


Mejora del MAE frente a la media movil 7 dias: 7.5% | IC95% [-5.3 %, 19.2 %]


### 7.2 Complemento: total del local y por categoria
En APF2 el docente indico que el dashboard no debia mostrar cada bebida y cada plato por separado, sino agrupados por **categoria**. Por eso, ademas del resultado principal (producto-dia), se evalua la prediccion **agregada**, donde el ruido se compensa. Es un analisis complementario: no reemplaza el resultado principal ni se usa para elegirlo.

Reglas fijadas antes de ver resultados: (1) todos los productos del periodo (no solo los 12); (2) una serie por categoria con al menos 500 unidades en el periodo (7 de 10 categorias; las otras 3 son demasiado pequenas); (3) mismos candidatos y mismo procedimiento que en 7.1: eleccion de alpha solo con validacion, reentrenamiento con entrenamiento + validacion y prueba (21 dias) usada una sola vez.

In [19]:
FT2 = ["l7", "m7", "esp"] + [f"d{i}" for i in range(7)]
def serie_a_tabla(s):
    q = s.to_frame("y"); q["l7"] = s.shift(7); q["m7"] = s.shift(1).rolling(7).mean()
    for i in range(7): q[f"d{i}"] = (q.index.dayofweek == i).astype(int)
    q["esp"] = q.index.isin(ESPECIALES).astype(int); return q.dropna()
def media_dia(fit, ev): return ev.index.dayofweek.map(fit.groupby(fit.index.dayofweek).y.mean()).values
def evaluar(s):
    global ULT
    q = serie_a_tabla(s); a_, b_, c_ = q[:"2026-08-30"], q["2026-08-31":"2026-09-20"], q["2026-09-21":]
    val = {al: mean_absolute_error(b_.y, Ridge(alpha=al).fit(a_[FT2], a_.y).predict(b_[FT2])) for al in [1, 10, 100]}; al = min(val, key=val.get)
    lleno = pd.concat([a_, b_]); pm = Ridge(alpha=al).fit(lleno[FT2], lleno.y).predict(c_[FT2])
    e_b, e_m = np.abs(c_.y.values - c_.m7.values), np.abs(c_.y.values - pm)
    ULT = (e_b, e_m, np.abs(c_.y.values - media_dia(lleno, c_)))
    return dict(u_dia=s.mean(), MAE_MA7=e_b.mean(), MAE_dia_sem=mean_absolute_error(c_.y, media_dia(lleno, c_)), MAE_Ridge=e_m.mean(),
                mejora_vs_MA7=1 - e_m.mean() / e_b.mean(), alpha_val=al, dias_gana=int((e_m < e_b).sum()), dias=len(e_m))
dias_t = pd.date_range("2026-04-12", "2026-10-11")
filas = {"TOTAL DEL LOCAL (92 productos)": evaluar(raw.groupby("fecha").unidades_vendidas.sum().reindex(dias_t, fill_value=0))}
E_TOT = ULT
for c, g in raw.groupby("categoria"):
    s_ = g.groupby("fecha").unidades_vendidas.sum().reindex(dias_t, fill_value=0)
    if s_.sum() >= 500: filas[c] = evaluar(s_)
agr = pd.DataFrame(filas).T
agr.to_csv("data/complemento_agregado.csv")
print("Unidades por categoria en el periodo:"); print(raw.groupby("categoria").unidades_vendidas.sum().sort_values(ascending=False).round(0).to_string())
print("\nRESULTADO EN PRUEBA (MAE en unidades por dia; 21 dias)"); print(agr.astype({"alpha_val": float}).round(3).to_string())

Unidades por categoria en el periodo:
categoria
Bebidas                           2369.0
Pollos a la Leña - Ofertas        1736.0
Platos Criollos y Guarniciones    1522.0
Tallarines y Sopas                 646.0
Promociones a la Parrilla          585.0
Pechugas y Anticuchos              584.0
Carnes a la Parrilla               524.0
Ofertas Especiales                 468.0
Menú para Niños                    262.0
Parrillas                          193.0

RESULTADO EN PRUEBA (MAE en unidades por dia; 21 dias)
                                 u_dia  MAE_MA7  MAE_dia_sem  MAE_Ridge  mejora_vs_MA7  alpha_val  dias_gana  dias
TOTAL DEL LOCAL (92 productos)  48.574    8.789        5.856      5.438          0.381        1.0       14.0  21.0
Bebidas                         12.945    3.306        3.070      3.052          0.077       10.0       13.0  21.0
Carnes a la Parrilla             2.863    1.776        1.787      1.802         -0.015        1.0       10.0  21.0
Pechugas y Anticuchos     

In [20]:
# Intervalo de confianza (bootstrap por dias, 5,000 remuestreos) para el total del local
e_b, e_m, e_d = E_TOT; rg = np.random.default_rng(SEED); idx = [rg.choice(len(e_m), len(e_m)) for _ in range(5000)]
for nom, ref in [("media movil 7 dias", e_b), ("media por dia de la semana", e_d)]:
    r_ = np.array([1 - e_m[i].mean() / ref[i].mean() for i in idx])
    print(f"Total del local, Ridge vs {nom}: mejora {1 - e_m.mean() / ref.mean():.1%} | IC95% [{np.percentile(r_, 2.5) * 100:.1f} %, {np.percentile(r_, 97.5) * 100:.1f} %]")

Total del local, Ridge vs media movil 7 dias: mejora 38.1% | IC95% [-3.7 %, 62.4 %]
Total del local, Ridge vs media por dia de la semana: mejora 7.1% | IC95% [-9.0 %, 19.7 %]


**Lectura (con las cifras impresas arriba):** al agregar, el error relativo baja y la mejora sobre la media movil sube, pero hay que leerla con cuidado. (1) Una parte grande viene de conocer el dia de la semana, algo que la media por dia de la semana ya captura; el modelo mejora poco sobre esa referencia. (2) La mejora no es uniforme: en las categorias pequenas casi no hay ganancia o es negativa. (3) Son solo 21 dias de prueba, por lo que los resultados son indicativos. (4) El total del local y las categorias responden a otro nivel de decision que el producto-dia: sirven para planificar compras generales y para el dashboard agrupado, no para decidir cuanto preparar de cada producto.

## 8. Interpretacion para el negocio

In [21]:
# Importancia de variables por permutacion sobre VALIDACION (el test no se usa para interpretar)
mval = mejores[ELEGIDO]
pi = permutation_importance(mval, Xva, yva, scoring="neg_mean_absolute_error", n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
imp.to_csv("data/importancia_variables.csv", header=["aumento_MAE_al_permutar"]); print(imp.round(4).to_string())
# Impacto estimado: unidades de error evitadas y su valor a precio de venta (referencial, no es un ahorro demostrado)
prec = raw.groupby("producto_id").apply(lambda g: g.monto_vendido.sum() / g.unidades_vendidas.sum())
n_prod, n_dias = te.producto_id.nunique(), te.fecha.nunique()
ev_dia = (mae_base - mae_mod) * n_prod
val_dia = float(((te2.assign(ec_base=(te.y - te.media_7).abs()).groupby("producto_id").apply(lambda g: (g.ec_base.mean() - g.abs_err.mean()) * prec[int(g.name)])).sum()))
print(f"\nUnidades de error evitadas por dia (12 productos): {ev_dia:.2f} | valor a precio de venta: S/ {val_dia:.2f} por dia ({n_dias} dias de test)")

producto_id       0.0412
media_14          0.0344
media_7           0.0093
dia_semana        0.0061
fin_de_semana     0.0030
lag_1             0.0004
lag_7             0.0003
fecha_especial    0.0000

Unidades de error evitadas por dia (12 productos): 0.77 | valor a precio de venta: S/ 6.15 por dia (21 dias de test)


## 9. Exportacion del modelo e integracion minima con el MVP

In [22]:
joblib.dump({"pipeline": final, "features": FEATS, "modelo": ELEGIDO, "random_state": SEED,
             "productos": {int(k): v for k, v in nombres.loc[top].items()}}, "model/modelo_demanda.joblib")

def predecir(registro: dict) -> dict:
    """Contrato de integracion: recibe JSON con campos que el MVP ya tiene en ventas/detalle_ventas."""
    art = joblib.load("model/modelo_demanda.joblib")
    f = pd.to_datetime(registro["fecha"])
    fila = pd.DataFrame([{"lag_1": registro["unidades_ayer"], "lag_7": registro["unidades_hace_7_dias"],
        "media_7": registro["media_7_dias"], "media_14": registro["media_14_dias"],
        "fin_de_semana": int(f.dayofweek in (5, 6)), "fecha_especial": int(f in ESPECIALES),
        "dia_semana": str(f.dayofweek), "producto_id": str(registro["producto_id"])}])[art["features"]]
    y = float(max(0.0, art["pipeline"].predict(fila)[0]))
    return {"producto_id": registro["producto_id"], "fecha": registro["fecha"], "unidades_predichas": round(y, 2),
            "unidades_sugeridas_compra": int(np.ceil(y)), "modelo": art["modelo"]}

ejemplo = te.iloc[0]
entrada = {"fecha": str(ejemplo.fecha.date()), "producto_id": int(ejemplo.producto_id), "unidades_ayer": float(ejemplo.lag_1),
           "unidades_hace_7_dias": float(ejemplo.lag_7), "media_7_dias": float(ejemplo.media_7), "media_14_dias": float(ejemplo.media_14)}
print("ENTRADA :", json.dumps(entrada, ensure_ascii=False)); salida = predecir(entrada)
print("SALIDA  :", json.dumps(salida, ensure_ascii=False), "| valor real ese dia:", float(ejemplo.y))
json.dump({"entrada": entrada, "salida": salida, "real": float(ejemplo.y)}, open("data/ejemplo_integracion.json", "w"), ensure_ascii=False, indent=2)
d.drop(columns=["dia_semana"]).assign(dia_semana=d.fecha.dt.dayofweek).to_csv("data/dataset_modelado.csv", index=False)
print("Modelo guardado en model/modelo_demanda.joblib")

ENTRADA : {"fecha": "2026-09-21", "producto_id": 3, "unidades_ayer": 3.0, "unidades_hace_7_dias": 1.0, "media_7_dias": 2.5714285714285716, "media_14_dias": 2.7857142857142856}
SALIDA  : {"producto_id": 3, "fecha": "2026-09-21", "unidades_predichas": 2.26, "unidades_sugeridas_compra": 3, "modelo": "Ridge (regresion regularizada)"} | valor real ese dia: 2.0
Modelo guardado en model/modelo_demanda.joblib
